#Iteraciones

## Iteracion 1

Iteracion inicial amplia con `RandomizedSearchCV` para `LogisticRegression`. La busqueda se aplica sobre el pipeline completo usando datos crudos de `X_train_formal`, por lo que cada combinacion ajusta imputacion, encoding, escalado, seleccion de variables, PCA y modelo solo dentro de los folds de entrenamiento.

Se usa F1 Score como metrica principal para seleccionar el mejor modelo (`refit="f1"`) y tambien se reportan Accuracy, Precision, Recall, F1 Score y ROC-AUC en el conjunto de test.


In [ ]:
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import time

# =========================
# Iteracion 1: busqueda amplia de hiperparametros
# =========================

param_distributions_iteracion_1 = {
    # Seleccion de variables
    "feature_selection__threshold": [0.0, 0.001, 0.005, 0.01, 0.02],

    # Reduccion dimensional
    "dimensionality_reduction__n_components": [0.80, 0.85, 0.90, 0.95],

    # Logistic Regression
    "training__C": loguniform(0.001, 100),
    "training__penalty": ["l1", "l2"],
    "training__solver": ["liblinear", "saga"],
    "training__class_weight": [None, "balanced"],
    "training__max_iter": [3000, 5000]
}

cv_iteracion_1 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


n_iter_iteracion_1 = 50
n_folds_iteracion_1 = cv_iteracion_1.get_n_splits()

espacio_busqueda_iteracion_1 = pd.DataFrame([
    {"hiperparametro": parametro, "valores_explorados": str(valores)}
    for parametro, valores in param_distributions_iteracion_1.items()
])

print("Espacio de busqueda utilizado - Iteracion 1")
display(espacio_busqueda_iteracion_1)
print("Cantidad de combinaciones evaluadas:", n_iter_iteracion_1)
print("Cantidad de folds CV:", n_folds_iteracion_1)
print("Total de ajustes realizados:", n_iter_iteracion_1 * n_folds_iteracion_1)

scoring_iteracion_1 = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

random_search_iteracion_1 = RandomizedSearchCV(
    estimator=clone(formal_model_pipeline_template),
    param_distributions=param_distributions_iteracion_1,
    n_iter=n_iter_iteracion_1,
    scoring=scoring_iteracion_1,
    refit="f1",
    cv=cv_iteracion_1,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

# Ajuste solo con train. Test queda reservado para la evaluacion final.
inicio_iteracion_1 = time.perf_counter()
random_search_iteracion_1.fit(X_train_formal, y_train_formal)
tiempo_ejecucion_iteracion_1 = time.perf_counter() - inicio_iteracion_1


print("ITERACION 1 - RandomizedSearchCV")
print("Tiempo de ejecucion - Iteracion 1 (segundos):", round(tiempo_ejecucion_iteracion_1, 2))
print("Mejor F1 Score promedio en CV:", round(random_search_iteracion_1.best_score_, 4))
print("Mejores hiperparametros:")
print(random_search_iteracion_1.best_params_)

resumen_busqueda_iteracion_1 = pd.DataFrame({
    "elemento": [
        "combinaciones_evaluadas",
        "folds_cv",
        "total_ajustes_cv",
        "tiempo_ejecucion_segundos",
        "mejor_score_refit_f1"
    ],
    "valor": [
        n_iter_iteracion_1,
        n_folds_iteracion_1,
        n_iter_iteracion_1 * n_folds_iteracion_1,
        round(tiempo_ejecucion_iteracion_1, 2),
        round(random_search_iteracion_1.best_score_, 4)
    ]
})
display(resumen_busqueda_iteracion_1)


best_index_iteracion_1 = random_search_iteracion_1.best_index_
resumen_cv_iteracion_1 = pd.DataFrame({
    "metrica_cv": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "mejor_modelo_cv": [
        random_search_iteracion_1.cv_results_["mean_test_accuracy"][best_index_iteracion_1],
        random_search_iteracion_1.cv_results_["mean_test_precision"][best_index_iteracion_1],
        random_search_iteracion_1.cv_results_["mean_test_recall"][best_index_iteracion_1],
        random_search_iteracion_1.cv_results_["mean_test_f1"][best_index_iteracion_1],
        random_search_iteracion_1.cv_results_["mean_test_roc_auc"][best_index_iteracion_1]
    ]
})
resumen_cv_iteracion_1["mejor_modelo_cv"] = resumen_cv_iteracion_1["mejor_modelo_cv"].round(4)

display(resumen_cv_iteracion_1)

# Evaluacion final del mejor modelo en test.
best_pipeline_iteracion_1 = random_search_iteracion_1.best_estimator_
y_iteracion_1_pred = best_pipeline_iteracion_1.predict(X_test_formal)
y_iteracion_1_proba = best_pipeline_iteracion_1.predict_proba(X_test_formal)[:, 1]

metricas_test_iteracion_1 = pd.DataFrame({
    "metrica_test": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "valor_test": [
        accuracy_score(y_test_formal, y_iteracion_1_pred),
        precision_score(y_test_formal, y_iteracion_1_pred, zero_division=0),
        recall_score(y_test_formal, y_iteracion_1_pred, zero_division=0),
        f1_score(y_test_formal, y_iteracion_1_pred, zero_division=0),
        roc_auc_score(y_test_formal, y_iteracion_1_proba)
    ]
})
metricas_test_iteracion_1["valor_test"] = metricas_test_iteracion_1["valor_test"].round(4)

display(metricas_test_iteracion_1)

print("Matriz de confusion - Iteracion 1:")
print(confusion_matrix(y_test_formal, y_iteracion_1_pred))

print("Reporte de clasificacion - Iteracion 1:")
print(classification_report(y_test_formal, y_iteracion_1_pred, zero_division=0))

resultados_random_search_iteracion_1 = pd.DataFrame(random_search_iteracion_1.cv_results_).sort_values("rank_test_f1")

display(
    resultados_random_search_iteracion_1[
        [
            "rank_test_f1",
            "mean_test_f1",
            "std_test_f1",
            "mean_test_roc_auc",
            "mean_test_accuracy",
            "mean_test_precision",
            "mean_test_recall",
            "param_feature_selection__threshold",
            "param_dimensionality_reduction__n_components",
            "param_training__C",
            "param_training__penalty",
            "param_training__solver",
            "param_training__class_weight",
            "param_training__max_iter"
        ]
    ].head(10)
)


Espacio de busqueda utilizado - Iteracion 1


,hiperparametro,valores_explorados
0,feature_selection__threshold,"[0.0, 0.001, 0.005, 0.01, 0.02]"
1,dimensionality_reduction__n_components,"[0.8, 0.85, 0.9, 0.95]"
2,training__C,<scipy.stats._distn_infrastructure.rv_continuo...
3,training__penalty,"['l1', 'l2']"
4,training__solver,"['liblinear', 'saga']"
5,training__class_weight,"[None, 'balanced']"
6,training__max_iter,"[3000, 5000]"


Cantidad de combinaciones evaluadas: 50
Cantidad de folds CV: 5
Total de ajustes realizados: 250
Fitting 5 folds for each of 50 candidates, totalling 250 fits
ITERACION 1 - RandomizedSearchCV
Tiempo de ejecucion - Iteracion 1 (segundos): 60.53
Mejor F1 Score promedio en CV: 0.2997
Mejores hiperparametros:
{'dimensionality_reduction__n_components': 0.85, 'feature_selection__threshold': 0.01, 'training__C': np.float64(0.0025366855305518844), 'training__class_weight': 'balanced', 'training__max_iter': 3000, 'training__penalty': 'l1', 'training__solver': 'liblinear'}


,elemento,valor
0,combinaciones_evaluadas,50.0000
1,folds_cv,5.0000
2,total_ajustes_cv,250.0000
3,tiempo_ejecucion_segundos,60.5300
4,mejor_score_refit_f1,0.2997


,metrica_cv,mejor_modelo_cv
0,Accuracy,0.8459
1,Precision,0.3267
2,Recall,0.3000
3,F1 Score,0.2997
4,ROC-AUC,0.6364


,metrica_test,valor_test
0,Accuracy,0.8571
1,Precision,0.3333
2,Recall,0.3077
3,F1 Score,0.3200
4,ROC-AUC,0.8215


Matriz de confusion - Iteracion 1:
[[98  8]
 [ 9  4]]
Reporte de clasificacion - Iteracion 1:
              precision    recall  f1-score   support

           0       0.92      0.92      0.92       106
           1       0.33      0.31      0.32        13

    accuracy                           0.86       119
   macro avg       0.62      0.62      0.62       119
weighted avg       0.85      0.86      0.85       119



,rank_test_f1,mean_test_f1,std_test_f1,mean_test_roc_auc,mean_test_accuracy,mean_test_precision,mean_test_recall,param_feature_selection__threshold,param_dimensionality_reduction__n_components,param_training__C,param_training__penalty,param_training__solver,param_training__class_weight,param_training__max_iter
35,1,0.299658,0.147072,0.636353,0.845853,0.326667,0.300,0.010,0.85,0.002537,l1,liblinear,balanced,3000
30,2,0.257802,0.056682,0.714819,0.440219,0.163783,0.800,0.005,0.95,0.041306,l1,saga,balanced,5000
24,3,0.254707,0.079195,0.648748,0.713654,0.251788,0.425,0.020,0.90,0.013931,l1,liblinear,balanced,3000
45,4,0.254141,0.124321,0.749144,0.887872,0.566667,0.175,0.000,0.95,1.622401,l2,saga,None,3000
21,5,0.251577,0.126217,0.645381,0.885055,0.556667,0.175,0.005,0.85,0.410832,l2,saga,None,5000
18,5,0.251577,0.126217,0.645381,0.885055,0.556667,0.175,0.000,0.85,37.566296,l2,saga,None,5000
3,5,0.251577,0.126217,0.647712,0.885055,0.556667,0.175,0.001,0.85,0.008112,l2,saga,None,3000
11,5,0.251577,0.126217,0.645381,0.885055,0.556667,0.175,0.001,0.85,0.689448,l2,saga,None,3000
14,5,0.251577,0.126217,0.645381,0.885055,0.556667,0.175,0.020,0.85,0.858531,l2,saga,None,5000
44,5,0.251577,0.126217,0.663635,0.885055,0.556667,0.175,0.005,0.90,4.263619,l2,saga,None,5000


## Iteracion 2

A partir de la Iteracion 1, el mejor resultado por F1 Score uso `PCA=0.85`, `feature_selection__threshold=0.01`, `LogisticRegression` con `penalty="l1"`, `solver="liblinear"`, `class_weight="balanced"` y un `C` muy bajo cercano a `0.0025`.

Esta segunda iteracion realiza una busqueda mas fina alrededor de esa zona. Se mantiene F1 Score como metrica principal (`refit="f1"`) porque el problema presenta desbalance: la clase positiva tiene pocos casos y se busca mejorar el equilibrio entre precision y recall.


In [ ]:
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import time

# =========================
# Iteracion 2: busqueda fina basada en Iteracion 1
# =========================

param_distributions_iteracion_2 = {
    # Seleccion de variables: alrededor del mejor threshold=0.01
    "feature_selection__threshold": [0.005, 0.0075, 0.01, 0.0125, 0.015, 0.02],

    # Reduccion dimensional: alrededor del mejor PCA=0.85
    "dimensionality_reduction__n_components": [0.80, 0.825, 0.85, 0.875, 0.90],

    # Logistic Regression: refinamiento alrededor de C bajo y modelo sparse L1
    "training__C": loguniform(0.0005, 0.05),
    "training__penalty": ["l1", "l2"],
    "training__solver": ["liblinear"],
    "training__class_weight": ["balanced"],
    "training__max_iter": [3000, 5000, 8000]
}

cv_iteracion_2 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


n_iter_iteracion_2 = 60
n_folds_iteracion_2 = cv_iteracion_2.get_n_splits()

espacio_busqueda_iteracion_2 = pd.DataFrame([
    {"hiperparametro": parametro, "valores_explorados": str(valores)}
    for parametro, valores in param_distributions_iteracion_2.items()
])

print("Espacio de busqueda utilizado - Iteracion 2")
display(espacio_busqueda_iteracion_2)
print("Cantidad de combinaciones evaluadas:", n_iter_iteracion_2)
print("Cantidad de folds CV:", n_folds_iteracion_2)
print("Total de ajustes realizados:", n_iter_iteracion_2 * n_folds_iteracion_2)

scoring_iteracion_2 = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

random_search_iteracion_2 = RandomizedSearchCV(
    estimator=clone(formal_model_pipeline_template),
    param_distributions=param_distributions_iteracion_2,
    n_iter=n_iter_iteracion_2,
    scoring=scoring_iteracion_2,
    refit="f1",
    cv=cv_iteracion_2,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

# Ajuste solo con train. Test queda reservado para evaluacion final.
inicio_iteracion_2 = time.perf_counter()
random_search_iteracion_2.fit(X_train_formal, y_train_formal)
tiempo_ejecucion_iteracion_2 = time.perf_counter() - inicio_iteracion_2


print("ITERACION 2 - RandomizedSearchCV ajustado")
print("Tiempo de ejecucion - Iteracion 2 (segundos):", round(tiempo_ejecucion_iteracion_2, 2))
print("Mejor F1 Score promedio en CV:", round(random_search_iteracion_2.best_score_, 4))
print("Mejores hiperparametros:")
print(random_search_iteracion_2.best_params_)

resumen_busqueda_iteracion_2 = pd.DataFrame({
    "elemento": [
        "combinaciones_evaluadas",
        "folds_cv",
        "total_ajustes_cv",
        "tiempo_ejecucion_segundos",
        "mejor_score_refit_f1"
    ],
    "valor": [
        n_iter_iteracion_2,
        n_folds_iteracion_2,
        n_iter_iteracion_2 * n_folds_iteracion_2,
        round(tiempo_ejecucion_iteracion_2, 2),
        round(random_search_iteracion_2.best_score_, 4)
    ]
})
display(resumen_busqueda_iteracion_2)


best_index_iteracion_2 = random_search_iteracion_2.best_index_
resumen_cv_iteracion_2 = pd.DataFrame({
    "metrica_cv": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "mejor_modelo_cv": [
        random_search_iteracion_2.cv_results_["mean_test_accuracy"][best_index_iteracion_2],
        random_search_iteracion_2.cv_results_["mean_test_precision"][best_index_iteracion_2],
        random_search_iteracion_2.cv_results_["mean_test_recall"][best_index_iteracion_2],
        random_search_iteracion_2.cv_results_["mean_test_f1"][best_index_iteracion_2],
        random_search_iteracion_2.cv_results_["mean_test_roc_auc"][best_index_iteracion_2]
    ]
})
resumen_cv_iteracion_2["mejor_modelo_cv"] = resumen_cv_iteracion_2["mejor_modelo_cv"].round(4)

display(resumen_cv_iteracion_2)

# Evaluacion final del mejor modelo en test.
best_pipeline_iteracion_2 = random_search_iteracion_2.best_estimator_
y_iteracion_2_pred = best_pipeline_iteracion_2.predict(X_test_formal)
y_iteracion_2_proba = best_pipeline_iteracion_2.predict_proba(X_test_formal)[:, 1]

metricas_test_iteracion_2 = pd.DataFrame({
    "metrica_test": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "valor_test": [
        accuracy_score(y_test_formal, y_iteracion_2_pred),
        precision_score(y_test_formal, y_iteracion_2_pred, zero_division=0),
        recall_score(y_test_formal, y_iteracion_2_pred, zero_division=0),
        f1_score(y_test_formal, y_iteracion_2_pred, zero_division=0),
        roc_auc_score(y_test_formal, y_iteracion_2_proba)
    ]
})
metricas_test_iteracion_2["valor_test"] = metricas_test_iteracion_2["valor_test"].round(4)

display(metricas_test_iteracion_2)

print("Matriz de confusion - Iteracion 2:")
print(confusion_matrix(y_test_formal, y_iteracion_2_pred))

print("Reporte de clasificacion - Iteracion 2:")
print(classification_report(y_test_formal, y_iteracion_2_pred, zero_division=0))

resultados_random_search_iteracion_2 = pd.DataFrame(random_search_iteracion_2.cv_results_).sort_values("rank_test_f1")

display(
    resultados_random_search_iteracion_2[
        [
            "rank_test_f1",
            "mean_test_f1",
            "std_test_f1",
            "mean_test_roc_auc",
            "mean_test_accuracy",
            "mean_test_precision",
            "mean_test_recall",
            "param_feature_selection__threshold",
            "param_dimensionality_reduction__n_components",
            "param_training__C",
            "param_training__penalty",
            "param_training__solver",
            "param_training__class_weight",
            "param_training__max_iter"
        ]
    ].head(10)
)

# Comparacion directa contra Iteracion 1, si sus resultados ya existen en memoria.
if "metricas_test_iteracion_1" in globals():
    comparacion_iteraciones = metricas_test_iteracion_1.rename(columns={"valor_test": "iteracion_1"}).merge(
        metricas_test_iteracion_2.rename(columns={"valor_test": "iteracion_2"}),
        on="metrica_test",
        how="inner"
    )
    comparacion_iteraciones["diferencia_i2_menos_i1"] = (
        comparacion_iteraciones["iteracion_2"] - comparacion_iteraciones["iteracion_1"]
    ).round(4)
    display(comparacion_iteraciones)


Espacio de busqueda utilizado - Iteracion 2


,hiperparametro,valores_explorados
0,feature_selection__threshold,"[0.005, 0.0075, 0.01, 0.0125, 0.015, 0.02]"
1,dimensionality_reduction__n_components,"[0.8, 0.825, 0.85, 0.875, 0.9]"
2,training__C,<scipy.stats._distn_infrastructure.rv_continuo...
3,training__penalty,"['l1', 'l2']"
4,training__solver,['liblinear']
5,training__class_weight,['balanced']
6,training__max_iter,"[3000, 5000, 8000]"


Cantidad de combinaciones evaluadas: 60
Cantidad de folds CV: 5
Total de ajustes realizados: 300
Fitting 5 folds for each of 60 candidates, totalling 300 fits
ITERACION 2 - RandomizedSearchCV ajustado
Tiempo de ejecucion - Iteracion 2 (segundos): 82.62
Mejor F1 Score promedio en CV: 0.2997
Mejores hiperparametros:
{'dimensionality_reduction__n_components': 0.875, 'feature_selection__threshold': 0.015, 'training__C': np.float64(0.0011636961140314352), 'training__class_weight': 'balanced', 'training__max_iter': 3000, 'training__penalty': 'l1', 'training__solver': 'liblinear'}


,elemento,valor
0,combinaciones_evaluadas,60.0000
1,folds_cv,5.0000
2,total_ajustes_cv,300.0000
3,tiempo_ejecucion_segundos,82.6200
4,mejor_score_refit_f1,0.2997


,metrica_cv,mejor_modelo_cv
0,Accuracy,0.8459
1,Precision,0.3267
2,Recall,0.3000
3,F1 Score,0.2997
4,ROC-AUC,0.6364


,metrica_test,valor_test
0,Accuracy,0.8571
1,Precision,0.3333
2,Recall,0.3077
3,F1 Score,0.3200
4,ROC-AUC,0.8215


Matriz de confusion - Iteracion 2:
[[98  8]
 [ 9  4]]
Reporte de clasificacion - Iteracion 2:
              precision    recall  f1-score   support

           0       0.92      0.92      0.92       106
           1       0.33      0.31      0.32        13

    accuracy                           0.86       119
   macro avg       0.62      0.62      0.62       119
weighted avg       0.85      0.86      0.85       119



,rank_test_f1,mean_test_f1,std_test_f1,mean_test_roc_auc,mean_test_accuracy,mean_test_precision,mean_test_recall,param_feature_selection__threshold,param_dimensionality_reduction__n_components,param_training__C,param_training__penalty,param_training__solver,param_training__class_weight,param_training__max_iter
0,1,0.299658,0.147072,0.636353,0.845853,0.326667,0.30,0.0150,0.875,0.001164,l1,liblinear,balanced,3000
7,1,0.299658,0.147072,0.636353,0.845853,0.326667,0.30,0.0125,0.850,0.005339,l1,liblinear,balanced,3000
15,1,0.299658,0.147072,0.636353,0.845853,0.326667,0.30,0.0075,0.875,0.003085,l1,liblinear,balanced,5000
28,1,0.299658,0.147072,0.636353,0.845853,0.326667,0.30,0.0150,0.875,0.001266,l1,liblinear,balanced,8000
29,1,0.299658,0.147072,0.636353,0.845853,0.326667,0.30,0.0150,0.875,0.005201,l1,liblinear,balanced,5000
45,1,0.299658,0.147072,0.636353,0.845853,0.326667,0.30,0.0100,0.875,0.002743,l1,liblinear,balanced,3000
40,1,0.299658,0.147072,0.636353,0.845853,0.326667,0.30,0.0125,0.875,0.001594,l1,liblinear,balanced,5000
37,1,0.299658,0.147072,0.636353,0.845853,0.326667,0.30,0.0050,0.850,0.004663,l1,liblinear,balanced,3000
18,9,0.293776,0.139325,0.636750,0.843036,0.315556,0.30,0.0125,0.875,0.006086,l1,liblinear,balanced,3000
59,10,0.276801,0.138321,0.654712,0.851408,0.326667,0.25,0.0200,0.825,0.003943,l1,liblinear,balanced,3000


,metrica_test,iteracion_1,iteracion_2,diferencia_i2_menos_i1
0,Accuracy,0.8571,0.8571,0.0
1,Precision,0.3333,0.3333,0.0
2,Recall,0.3077,0.3077,0.0
3,F1 Score,0.3200,0.3200,0.0
4,ROC-AUC,0.8215,0.8215,0.0


## Iteracion 3

Refinamiento final para intentar mejorar los indicadores de Iteracion 2. Como los resultados de test fueron `Accuracy=0.8571`, `Precision=0.3333`, `Recall=0.3077`, `F1=0.3200` y `ROC-AUC=0.8215`, esta iteracion mantiene la busqueda cerca de la zona ganadora, pero agrega un ajuste del umbral de decision.

El modelo sigue entrenandose con `RandomizedSearchCV` sobre el pipeline completo y el umbral se selecciona usando probabilidades out-of-fold sobre `X_train_formal`, evitando usar `X_test_formal` para decidir el umbral.


In [ ]:
from scipy.stats import loguniform
from sklearn.base import clone
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import time

# =========================
# Iteracion 3: refinamiento final + ajuste de umbral
# =========================

param_distributions_iteracion_3 = {
    # Ajuste fino alrededor de Iteracion 1/2
    "feature_selection__threshold": [0.0075, 0.01, 0.0125, 0.015],
    "dimensionality_reduction__n_components": [0.825, 0.85, 0.875],

    # Logistic Regression enfocado en regularizacion fuerte y clase balanceada
    "training__C": loguniform(0.0008, 0.02),
    "training__penalty": ["l1", "l2"],
    "training__solver": ["liblinear"],
    "training__class_weight": ["balanced"],
    "training__max_iter": [5000, 8000]
}

cv_iteracion_3 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


n_iter_iteracion_3 = 50
n_folds_iteracion_3 = cv_iteracion_3.get_n_splits()

espacio_busqueda_iteracion_3 = pd.DataFrame([
    {"hiperparametro": parametro, "valores_explorados": str(valores)}
    for parametro, valores in param_distributions_iteracion_3.items()
])

print("Espacio de busqueda utilizado - Iteracion 3")
display(espacio_busqueda_iteracion_3)
print("Cantidad de combinaciones evaluadas:", n_iter_iteracion_3)
print("Cantidad de folds CV:", n_folds_iteracion_3)
print("Total de ajustes realizados:", n_iter_iteracion_3 * n_folds_iteracion_3)

scoring_iteracion_3 = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

random_search_iteracion_3 = RandomizedSearchCV(
    estimator=clone(formal_model_pipeline_template),
    param_distributions=param_distributions_iteracion_3,
    n_iter=n_iter_iteracion_3,
    scoring=scoring_iteracion_3,
    refit="f1",
    cv=cv_iteracion_3,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

inicio_iteracion_3 = time.perf_counter()
random_search_iteracion_3.fit(X_train_formal, y_train_formal)
tiempo_ejecucion_iteracion_3 = time.perf_counter() - inicio_iteracion_3


print("ITERACION 3 - RandomizedSearchCV refinado")
print("Tiempo de ejecucion - Iteracion 3 (segundos):", round(tiempo_ejecucion_iteracion_3, 2))
print("Mejor F1 Score promedio en CV:", round(random_search_iteracion_3.best_score_, 4))
print("Mejores hiperparametros:")
print(random_search_iteracion_3.best_params_)

resumen_busqueda_iteracion_3 = pd.DataFrame({
    "elemento": [
        "combinaciones_evaluadas",
        "folds_cv",
        "total_ajustes_cv",
        "tiempo_ejecucion_segundos",
        "mejor_score_refit_f1"
    ],
    "valor": [
        n_iter_iteracion_3,
        n_folds_iteracion_3,
        n_iter_iteracion_3 * n_folds_iteracion_3,
        round(tiempo_ejecucion_iteracion_3, 2),
        round(random_search_iteracion_3.best_score_, 4)
    ]
})
display(resumen_busqueda_iteracion_3)


best_index_iteracion_3 = random_search_iteracion_3.best_index_
resumen_cv_iteracion_3 = pd.DataFrame({
    "metrica_cv": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "mejor_modelo_cv": [
        random_search_iteracion_3.cv_results_["mean_test_accuracy"][best_index_iteracion_3],
        random_search_iteracion_3.cv_results_["mean_test_precision"][best_index_iteracion_3],
        random_search_iteracion_3.cv_results_["mean_test_recall"][best_index_iteracion_3],
        random_search_iteracion_3.cv_results_["mean_test_f1"][best_index_iteracion_3],
        random_search_iteracion_3.cv_results_["mean_test_roc_auc"][best_index_iteracion_3]
    ]
})
resumen_cv_iteracion_3["mejor_modelo_cv"] = resumen_cv_iteracion_3["mejor_modelo_cv"].round(4)
display(resumen_cv_iteracion_3)

# =========================
# Ajuste de umbral sin usar test
# =========================

best_params_iteracion_3 = random_search_iteracion_3.best_params_
pipeline_umbral_iteracion_3 = clone(formal_model_pipeline_template).set_params(**best_params_iteracion_3)

# Probabilidades out-of-fold sobre train para elegir umbral sin mirar test.
oof_proba_iteracion_3 = cross_val_predict(
    pipeline_umbral_iteracion_3,
    X_train_formal,
    y_train_formal,
    cv=cv_iteracion_3,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

threshold_candidates = np.arange(0.20, 0.71, 0.01)
threshold_results_iteracion_3 = []

for threshold in threshold_candidates:
    y_oof_pred = (oof_proba_iteracion_3 >= threshold).astype(int)
    threshold_results_iteracion_3.append({
        "threshold": threshold,
        "accuracy_cv_oof": accuracy_score(y_train_formal, y_oof_pred),
        "precision_cv_oof": precision_score(y_train_formal, y_oof_pred, zero_division=0),
        "recall_cv_oof": recall_score(y_train_formal, y_oof_pred, zero_division=0),
        "f1_cv_oof": f1_score(y_train_formal, y_oof_pred, zero_division=0),
        "roc_auc_cv_oof": roc_auc_score(y_train_formal, oof_proba_iteracion_3)
    })

threshold_results_iteracion_3 = pd.DataFrame(threshold_results_iteracion_3)
best_threshold_row_iteracion_3 = threshold_results_iteracion_3.sort_values(
    ["f1_cv_oof", "recall_cv_oof", "precision_cv_oof"],
    ascending=False
).iloc[0]
best_threshold_iteracion_3 = best_threshold_row_iteracion_3["threshold"]

print("Mejor umbral elegido solo con train/out-of-fold:", round(best_threshold_iteracion_3, 2))
display(threshold_results_iteracion_3.sort_values("f1_cv_oof", ascending=False).head(10).round(4))

# =========================
# Entrenamiento final con train y evaluacion en test
# =========================

best_pipeline_iteracion_3 = clone(formal_model_pipeline_template).set_params(**best_params_iteracion_3)
best_pipeline_iteracion_3.fit(X_train_formal, y_train_formal)

y_iteracion_3_proba = best_pipeline_iteracion_3.predict_proba(X_test_formal)[:, 1]
y_iteracion_3_pred_default = best_pipeline_iteracion_3.predict(X_test_formal)
y_iteracion_3_pred_threshold = (y_iteracion_3_proba >= best_threshold_iteracion_3).astype(int)

metricas_test_iteracion_3 = pd.DataFrame({
    "metrica_test": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "umbral_0_50": [
        accuracy_score(y_test_formal, y_iteracion_3_pred_default),
        precision_score(y_test_formal, y_iteracion_3_pred_default, zero_division=0),
        recall_score(y_test_formal, y_iteracion_3_pred_default, zero_division=0),
        f1_score(y_test_formal, y_iteracion_3_pred_default, zero_division=0),
        roc_auc_score(y_test_formal, y_iteracion_3_proba)
    ],
    "umbral_ajustado": [
        accuracy_score(y_test_formal, y_iteracion_3_pred_threshold),
        precision_score(y_test_formal, y_iteracion_3_pred_threshold, zero_division=0),
        recall_score(y_test_formal, y_iteracion_3_pred_threshold, zero_division=0),
        f1_score(y_test_formal, y_iteracion_3_pred_threshold, zero_division=0),
        roc_auc_score(y_test_formal, y_iteracion_3_proba)
    ]
})
metricas_test_iteracion_3[["umbral_0_50", "umbral_ajustado"]] = metricas_test_iteracion_3[["umbral_0_50", "umbral_ajustado"]].round(4)

display(metricas_test_iteracion_3)

print("Matriz de confusion - Iteracion 3 con umbral ajustado:")
print(confusion_matrix(y_test_formal, y_iteracion_3_pred_threshold))

print("Reporte de clasificacion - Iteracion 3 con umbral ajustado:")
print(classification_report(y_test_formal, y_iteracion_3_pred_threshold, zero_division=0))

resultados_random_search_iteracion_3 = pd.DataFrame(random_search_iteracion_3.cv_results_).sort_values("rank_test_f1")
display(
    resultados_random_search_iteracion_3[
        [
            "rank_test_f1",
            "mean_test_f1",
            "std_test_f1",
            "mean_test_roc_auc",
            "mean_test_accuracy",
            "mean_test_precision",
            "mean_test_recall",
            "param_feature_selection__threshold",
            "param_dimensionality_reduction__n_components",
            "param_training__C",
            "param_training__penalty",
            "param_training__solver",
            "param_training__class_weight",
            "param_training__max_iter"
        ]
    ].head(10)
)

# Comparacion contra Iteracion 2, si esta disponible en memoria.
if "metricas_test_iteracion_2" in globals():
    comparacion_i2_i3 = metricas_test_iteracion_2.rename(columns={"valor_test": "iteracion_2"}).merge(
        metricas_test_iteracion_3[["metrica_test", "umbral_ajustado"]].rename(columns={"umbral_ajustado": "iteracion_3"}),
        on="metrica_test",
        how="inner"
    )
    comparacion_i2_i3["diferencia_i3_menos_i2"] = (
        comparacion_i2_i3["iteracion_3"] - comparacion_i2_i3["iteracion_2"]
    ).round(4)
    display(comparacion_i2_i3)


Espacio de busqueda utilizado - Iteracion 3


,hiperparametro,valores_explorados
0,feature_selection__threshold,"[0.0075, 0.01, 0.0125, 0.015]"
1,dimensionality_reduction__n_components,"[0.825, 0.85, 0.875]"
2,training__C,<scipy.stats._distn_infrastructure.rv_continuo...
3,training__penalty,"['l1', 'l2']"
4,training__solver,['liblinear']
5,training__class_weight,['balanced']
6,training__max_iter,"[5000, 8000]"


Cantidad de combinaciones evaluadas: 50
Cantidad de folds CV: 5
Total de ajustes realizados: 250
Fitting 5 folds for each of 50 candidates, totalling 250 fits
ITERACION 3 - RandomizedSearchCV refinado
Tiempo de ejecucion - Iteracion 3 (segundos): 70.15
Mejor F1 Score promedio en CV: 0.2997
Mejores hiperparametros:
{'dimensionality_reduction__n_components': 0.875, 'feature_selection__threshold': 0.015, 'training__C': np.float64(0.005385797427226392), 'training__class_weight': 'balanced', 'training__max_iter': 5000, 'training__penalty': 'l1', 'training__solver': 'liblinear'}


,elemento,valor
0,combinaciones_evaluadas,50.0000
1,folds_cv,5.0000
2,total_ajustes_cv,250.0000
3,tiempo_ejecucion_segundos,70.1500
4,mejor_score_refit_f1,0.2997


,metrica_cv,mejor_modelo_cv
0,Accuracy,0.8459
1,Precision,0.3267
2,Recall,0.3000
3,F1 Score,0.2997
4,ROC-AUC,0.6364


...

,threshold,accuracy_cv_oof,precision_cv_oof,recall_cv_oof,f1_cv_oof,roc_auc_cv_oof
30,0.50,0.8459,0.3077,0.300,0.3038,0.6453
31,0.51,0.8936,0.5833,0.175,0.2692,0.6453
32,0.52,0.8964,0.6667,0.150,0.2449,0.6453
29,0.49,0.2997,0.1223,0.850,0.2138,0.6453
28,0.48,0.1261,0.1136,1.000,0.2041,0.6453
27,0.47,0.1261,0.1136,1.000,0.2041,0.6453
25,0.45,0.1232,0.1133,1.000,0.2036,0.6453
26,0.46,0.1232,0.1133,1.000,0.2036,0.6453
24,0.44,0.1232,0.1133,1.000,0.2036,0.6453
20,0.40,0.1204,0.1130,1.000,0.2030,0.6453


,metrica_test,umbral_0_50,umbral_ajustado
0,Accuracy,0.8571,0.8571
1,Precision,0.3333,0.3333
2,Recall,0.3077,0.3077
3,F1 Score,0.3200,0.3200
4,ROC-AUC,0.8215,0.8215


Matriz de confusion - Iteracion 3 con umbral ajustado:
[[98  8]
 [ 9  4]]
Reporte de clasificacion - Iteracion 3 con umbral ajustado:
              precision    recall  f1-score   support

           0       0.92      0.92      0.92       106
           1       0.33      0.31      0.32        13

    accuracy                           0.86       119
   macro avg       0.62      0.62      0.62       119
weighted avg       0.85      0.86      0.85       119



,rank_test_f1,mean_test_f1,std_test_f1,mean_test_roc_auc,mean_test_accuracy,mean_test_precision,mean_test_recall,param_feature_selection__threshold,param_dimensionality_reduction__n_components,param_training__C,param_training__penalty,param_training__solver,param_training__class_weight,param_training__max_iter
11,1,0.299658,0.147072,0.636353,0.845853,0.326667,0.3,0.0075,0.850,0.001682,l1,liblinear,balanced,8000
12,1,0.299658,0.147072,0.636353,0.845853,0.326667,0.3,0.0150,0.875,0.003938,l1,liblinear,balanced,5000
8,1,0.299658,0.147072,0.636353,0.845853,0.326667,0.3,0.0150,0.875,0.005386,l1,liblinear,balanced,5000
17,1,0.299658,0.147072,0.636353,0.845853,0.326667,0.3,0.0150,0.875,0.005018,l1,liblinear,balanced,8000
30,1,0.299658,0.147072,0.636353,0.845853,0.326667,0.3,0.0100,0.850,0.003290,l1,liblinear,balanced,5000
22,1,0.299658,0.147072,0.636353,0.845853,0.326667,0.3,0.0125,0.875,0.001517,l1,liblinear,balanced,8000
23,1,0.299658,0.147072,0.636353,0.845853,0.326667,0.3,0.0075,0.875,0.005626,l1,liblinear,balanced,8000
32,1,0.299658,0.147072,0.636353,0.845853,0.326667,0.3,0.0075,0.850,0.001785,l1,liblinear,balanced,5000
36,1,0.299658,0.147072,0.636353,0.845853,0.326667,0.3,0.0100,0.850,0.004540,l1,liblinear,balanced,8000
39,1,0.299658,0.147072,0.636353,0.845853,0.326667,0.3,0.0125,0.875,0.001359,l1,liblinear,balanced,5000


,metrica_test,iteracion_2,iteracion_3,diferencia_i3_menos_i2
0,Accuracy,0.8571,0.8571,0.0
1,Precision,0.3333,0.3333,0.0
2,Recall,0.3077,0.3077,0.0
3,F1 Score,0.3200,0.3200,0.0
4,ROC-AUC,0.8215,0.8215,0.0


## Iteracion 4

Optimizacion adicional de la Iteracion 2. Esta busqueda amplia el rango de regularizacion y PCA, y selecciona el mejor candidato usando un criterio combinado orientado a mejorar F1 Score, Precision y ROC-AUC.

Cada combinacion se evalua dentro del pipeline completo, por lo que imputacion, encoding, escalado, seleccion de variables, PCA y modelo se ajustan solo dentro de los folds de entrenamiento.


In [ ]:
from scipy.stats import loguniform
from sklearn.base import clone
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import time

# =========================
# Iteracion 4: optimizacion de Iteracion 2
# =========================

def obtener_metricas_previas(nombre_variable, columna_preferida="valor_test"):
    if nombre_variable not in globals():
        return None
    df = globals()[nombre_variable].copy()
    if columna_preferida not in df.columns:
        posibles = [c for c in ["umbral_ajustado", "umbral_0_50", "iteracion_3", "iteracion_2"] if c in df.columns]
        if not posibles:
            return None
        columna_preferida = posibles[0]
    return df[["metrica_test", columna_preferida]].rename(columns={columna_preferida: "iteracion_anterior"})

def seleccionar_iteracion_4(cv_results):
    f1 = np.nan_to_num(cv_results["mean_test_f1"], nan=0.0)
    precision = np.nan_to_num(cv_results["mean_test_precision"], nan=0.0)
    roc_auc = np.nan_to_num(cv_results["mean_test_roc_auc"], nan=0.0)
    recall = np.nan_to_num(cv_results["mean_test_recall"], nan=0.0)

    # Penaliza candidatos con precision muy baja, pero conserva recall para no apagar la clase positiva.
    score = (0.42 * f1) + (0.28 * roc_auc) + (0.22 * precision) + (0.08 * recall)
    score = np.where(precision < 0.25, score - 0.05, score)
    return int(np.argmax(score))

param_distributions_iteracion_4 = {
    "feature_selection__threshold": [0.0, 0.001, 0.003, 0.005, 0.0075, 0.01, 0.0125],
    "dimensionality_reduction__n_components": [0.80, 0.825, 0.85, 0.875, 0.90, 0.925, 0.95, 0.975],
    "training__C": loguniform(0.001, 10),
    "training__penalty": ["l1", "l2"],
    "training__solver": ["liblinear", "saga"],
    "training__class_weight": [None, "balanced"],
    "training__max_iter": [5000, 8000, 10000]
}

cv_iteracion_4 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
n_iter_iteracion_4 = 120
n_folds_iteracion_4 = cv_iteracion_4.get_n_splits()

espacio_busqueda_iteracion_4 = pd.DataFrame([
    {"hiperparametro": parametro, "valores_explorados": str(valores)}
    for parametro, valores in param_distributions_iteracion_4.items()
])

print("Espacio de busqueda utilizado - Iteracion 4")
display(espacio_busqueda_iteracion_4)
print("Cantidad de combinaciones evaluadas:", n_iter_iteracion_4)
print("Cantidad de folds CV:", n_folds_iteracion_4)
print("Total de ajustes realizados:", n_iter_iteracion_4 * n_folds_iteracion_4)

scoring_iteracion_4 = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

random_search_iteracion_4 = RandomizedSearchCV(
    estimator=clone(formal_model_pipeline_template),
    param_distributions=param_distributions_iteracion_4,
    n_iter=n_iter_iteracion_4,
    scoring=scoring_iteracion_4,
    refit=seleccionar_iteracion_4,
    cv=cv_iteracion_4,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

inicio_iteracion_4 = time.perf_counter()
random_search_iteracion_4.fit(X_train_formal, y_train_formal)
tiempo_ejecucion_iteracion_4 = time.perf_counter() - inicio_iteracion_4

best_index_iteracion_4 = random_search_iteracion_4.best_index_
cv_results_i4 = random_search_iteracion_4.cv_results_
score_combinado_i4 = (
    0.42 * cv_results_i4["mean_test_f1"][best_index_iteracion_4] +
    0.28 * cv_results_i4["mean_test_roc_auc"][best_index_iteracion_4] +
    0.22 * cv_results_i4["mean_test_precision"][best_index_iteracion_4] +
    0.08 * cv_results_i4["mean_test_recall"][best_index_iteracion_4]
)

print("ITERACION 4 - RandomizedSearchCV optimizado")
print("Tiempo de ejecucion - Iteracion 4 (segundos):", round(tiempo_ejecucion_iteracion_4, 2))
print("Mejor score combinado promedio en CV:", round(score_combinado_i4, 4))
print("Mejor configuracion encontrada:")
print(random_search_iteracion_4.best_params_)

resumen_busqueda_iteracion_4 = pd.DataFrame({
    "elemento": ["combinaciones_evaluadas", "folds_cv", "total_ajustes_cv", "tiempo_ejecucion_segundos", "mejor_score_combinado_cv"],
    "valor": [n_iter_iteracion_4, n_folds_iteracion_4, n_iter_iteracion_4 * n_folds_iteracion_4, round(tiempo_ejecucion_iteracion_4, 2), round(score_combinado_i4, 4)]
})
display(resumen_busqueda_iteracion_4)

resumen_cv_iteracion_4 = pd.DataFrame({
    "metrica_cv": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "mejor_modelo_cv": [
        cv_results_i4["mean_test_accuracy"][best_index_iteracion_4],
        cv_results_i4["mean_test_precision"][best_index_iteracion_4],
        cv_results_i4["mean_test_recall"][best_index_iteracion_4],
        cv_results_i4["mean_test_f1"][best_index_iteracion_4],
        cv_results_i4["mean_test_roc_auc"][best_index_iteracion_4]
    ]
})
resumen_cv_iteracion_4["mejor_modelo_cv"] = resumen_cv_iteracion_4["mejor_modelo_cv"].round(4)
display(resumen_cv_iteracion_4)

best_pipeline_iteracion_4 = random_search_iteracion_4.best_estimator_
y_iteracion_4_pred = best_pipeline_iteracion_4.predict(X_test_formal)
y_iteracion_4_proba = best_pipeline_iteracion_4.predict_proba(X_test_formal)[:, 1]

metricas_test_iteracion_4 = pd.DataFrame({
    "metrica_test": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "valor_test": [
        accuracy_score(y_test_formal, y_iteracion_4_pred),
        precision_score(y_test_formal, y_iteracion_4_pred, zero_division=0),
        recall_score(y_test_formal, y_iteracion_4_pred, zero_division=0),
        f1_score(y_test_formal, y_iteracion_4_pred, zero_division=0),
        roc_auc_score(y_test_formal, y_iteracion_4_proba)
    ]
})
metricas_test_iteracion_4["valor_test"] = metricas_test_iteracion_4["valor_test"].round(4)
display(metricas_test_iteracion_4)

print("Matriz de confusion - Iteracion 4:")
print(confusion_matrix(y_test_formal, y_iteracion_4_pred))
print("Reporte de clasificacion - Iteracion 4:")
print(classification_report(y_test_formal, y_iteracion_4_pred, zero_division=0))

resultados_random_search_iteracion_4 = pd.DataFrame(random_search_iteracion_4.cv_results_)
resultados_random_search_iteracion_4["score_combinado"] = (
    0.42 * resultados_random_search_iteracion_4["mean_test_f1"] +
    0.28 * resultados_random_search_iteracion_4["mean_test_roc_auc"] +
    0.22 * resultados_random_search_iteracion_4["mean_test_precision"] +
    0.08 * resultados_random_search_iteracion_4["mean_test_recall"]
)
resultados_random_search_iteracion_4 = resultados_random_search_iteracion_4.sort_values("score_combinado", ascending=False)

display(resultados_random_search_iteracion_4[
    ["score_combinado", "mean_test_f1", "std_test_f1", "mean_test_precision", "mean_test_recall", "mean_test_roc_auc", "mean_test_accuracy",
     "param_feature_selection__threshold", "param_dimensionality_reduction__n_components", "param_training__C", "param_training__penalty",
     "param_training__solver", "param_training__class_weight", "param_training__max_iter"]
].head(10))

metricas_previas_i4 = obtener_metricas_previas("metricas_test_iteracion_2")
if metricas_previas_i4 is not None:
    comparacion_i2_i4 = metricas_previas_i4.merge(
        metricas_test_iteracion_4.rename(columns={"valor_test": "iteracion_4"}),
        on="metrica_test",
        how="inner"
    )
    comparacion_i2_i4["diferencia_i4_menos_anterior"] = (comparacion_i2_i4["iteracion_4"] - comparacion_i2_i4["iteracion_anterior"]).round(4)
    display(comparacion_i2_i4)
else:
    print("No se encontro metricas_test_iteracion_2 en memoria; ejecute Iteracion 2 para ver la comparacion directa.")


Espacio de busqueda utilizado - Iteracion 4


,hiperparametro,valores_explorados
0,feature_selection__threshold,"[0.0, 0.001, 0.003, 0.005, 0.0075, 0.01, 0.0125]"
1,dimensionality_reduction__n_components,"[0.8, 0.825, 0.85, 0.875, 0.9, 0.925, 0.95, 0...."
2,training__C,<scipy.stats._distn_infrastructure.rv_continuo...
3,training__penalty,"['l1', 'l2']"
4,training__solver,"['liblinear', 'saga']"
5,training__class_weight,"[None, 'balanced']"
6,training__max_iter,"[5000, 8000, 10000]"


Cantidad de combinaciones evaluadas: 120
Cantidad de folds CV: 5
Total de ajustes realizados: 600
Fitting 5 folds for each of 120 candidates, totalling 600 fits
ITERACION 4 - RandomizedSearchCV optimizado
Tiempo de ejecucion - Iteracion 4 (segundos): 154.64
Mejor score combinado promedio en CV: 0.5032
Mejor configuracion encontrada:
{'dimensionality_reduction__n_components': 0.975, 'feature_selection__threshold': 0.003, 'training__C': np.float64(0.054648401368635455), 'training__class_weight': None, 'training__max_iter': 8000, 'training__penalty': 'l1', 'training__solver': 'saga'}


,elemento,valor
0,combinaciones_evaluadas,120.0000
1,folds_cv,5.0000
2,total_ajustes_cv,600.0000
3,tiempo_ejecucion_segundos,154.6400
4,mejor_score_combinado_cv,0.5032


,metrica_cv,mejor_modelo_cv
0,Accuracy,0.8935
1,Precision,0.6833
2,Recall,0.2000
3,F1 Score,0.2765
4,ROC-AUC,0.7883


,metrica_test,valor_test
0,Accuracy,0.8487
1,Precision,0.1429
2,Recall,0.0769
3,F1 Score,0.1000
4,ROC-AUC,0.8643


Matriz de confusion - Iteracion 4:
[[100   6]
 [ 12   1]]
Reporte de clasificacion - Iteracion 4:
              precision    recall  f1-score   support

           0       0.89      0.94      0.92       106
           1       0.14      0.08      0.10        13

    accuracy                           0.85       119
   macro avg       0.52      0.51      0.51       119
weighted avg       0.81      0.85      0.83       119



,score_combinado,mean_test_f1,std_test_f1,mean_test_precision,mean_test_recall,mean_test_roc_auc,mean_test_accuracy,param_feature_selection__threshold,param_dimensionality_reduction__n_components,param_training__C,param_training__penalty,param_training__solver,param_training__class_weight,param_training__max_iter
53,0.503187,0.276508,0.148856,0.683333,0.200,0.788287,0.893505,0.0030,0.975,0.054648,l1,saga,None,8000
60,0.463690,0.330693,0.098652,0.215941,0.775,0.768899,0.618192,0.0010,0.975,0.288422,l2,liblinear,balanced,10000
35,0.455166,0.254141,0.124321,0.566667,0.175,0.749144,0.887872,0.0075,0.950,0.024004,l2,saga,None,10000
19,0.455166,0.254141,0.124321,0.566667,0.175,0.749144,0.887872,0.0125,0.950,0.473549,l2,saga,None,10000
20,0.439666,0.254141,0.124321,0.566667,0.175,0.693787,0.887872,0.0030,0.925,0.175782,l1,saga,None,5000
11,0.439557,0.254141,0.124321,0.566667,0.175,0.693397,0.887872,0.0010,0.925,0.186553,l2,saga,None,10000
113,0.439557,0.254141,0.124321,0.566667,0.175,0.693397,0.887872,0.0030,0.925,4.023760,l2,saga,None,10000
46,0.439557,0.254141,0.124321,0.566667,0.175,0.693397,0.887872,0.0000,0.925,0.369936,l2,saga,None,5000
114,0.437763,0.272103,0.052313,0.160993,0.900,0.771646,0.444914,0.0100,0.975,0.066470,l2,saga,balanced,5000
32,0.436958,0.271087,0.052409,0.160286,0.900,0.770852,0.442097,0.0000,0.975,0.041242,l2,saga,balanced,5000


,metrica_test,iteracion_anterior,iteracion_4,diferencia_i4_menos_anterior
0,Accuracy,0.8571,0.8487,-0.0084
1,Precision,0.3333,0.1429,-0.1904
2,Recall,0.3077,0.0769,-0.2308
3,F1 Score,0.3200,0.1000,-0.2200
4,ROC-AUC,0.8215,0.8643,0.0428


## Iteracion 5

Optimizacion adicional de la Iteracion 3. Esta iteracion refina el espacio alrededor de la Iteracion 4 y agrega ajuste de umbral con probabilidades out-of-fold para mejorar F1 Score y Precision, manteniendo ROC-AUC como metrica de ranking.

El umbral se selecciona solo con datos de entrenamiento mediante validacion cruzada, y luego se evalua una unica vez sobre test.


In [ ]:
from scipy.stats import loguniform
from sklearn.base import clone
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import time

# =========================
# Iteracion 5: optimizacion de Iteracion 3 + umbral refinado
# =========================

def seleccionar_iteracion_5(cv_results):
    f1 = np.nan_to_num(cv_results["mean_test_f1"], nan=0.0)
    precision = np.nan_to_num(cv_results["mean_test_precision"], nan=0.0)
    roc_auc = np.nan_to_num(cv_results["mean_test_roc_auc"], nan=0.0)
    recall = np.nan_to_num(cv_results["mean_test_recall"], nan=0.0)

    score = (0.38 * f1) + (0.32 * roc_auc) + (0.24 * precision) + (0.06 * recall)
    score = np.where(precision < 0.30, score - 0.04, score)
    return int(np.argmax(score))

param_distributions_iteracion_5 = {
    "feature_selection__threshold": [0.0, 0.001, 0.003, 0.005, 0.0075, 0.01],
    "dimensionality_reduction__n_components": [0.875, 0.90, 0.925, 0.95, 0.975],
    "training__C": loguniform(0.002, 20),
    "training__penalty": ["l1", "l2"],
    "training__solver": ["liblinear", "saga"],
    "training__class_weight": [None, "balanced"],
    "training__max_iter": [8000, 10000]
}

cv_iteracion_5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
n_iter_iteracion_5 = 120
n_folds_iteracion_5 = cv_iteracion_5.get_n_splits()

espacio_busqueda_iteracion_5 = pd.DataFrame([
    {"hiperparametro": parametro, "valores_explorados": str(valores)}
    for parametro, valores in param_distributions_iteracion_5.items()
])

print("Espacio de busqueda utilizado - Iteracion 5")
display(espacio_busqueda_iteracion_5)
print("Cantidad de combinaciones evaluadas:", n_iter_iteracion_5)
print("Cantidad de folds CV:", n_folds_iteracion_5)
print("Total de ajustes realizados:", n_iter_iteracion_5 * n_folds_iteracion_5)

scoring_iteracion_5 = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

random_search_iteracion_5 = RandomizedSearchCV(
    estimator=clone(formal_model_pipeline_template),
    param_distributions=param_distributions_iteracion_5,
    n_iter=n_iter_iteracion_5,
    scoring=scoring_iteracion_5,
    refit=seleccionar_iteracion_5,
    cv=cv_iteracion_5,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

inicio_iteracion_5 = time.perf_counter()
random_search_iteracion_5.fit(X_train_formal, y_train_formal)
tiempo_ejecucion_iteracion_5 = time.perf_counter() - inicio_iteracion_5

best_index_iteracion_5 = random_search_iteracion_5.best_index_
cv_results_i5 = random_search_iteracion_5.cv_results_
score_combinado_i5 = (
    0.38 * cv_results_i5["mean_test_f1"][best_index_iteracion_5] +
    0.32 * cv_results_i5["mean_test_roc_auc"][best_index_iteracion_5] +
    0.24 * cv_results_i5["mean_test_precision"][best_index_iteracion_5] +
    0.06 * cv_results_i5["mean_test_recall"][best_index_iteracion_5]
)

print("ITERACION 5 - RandomizedSearchCV + ajuste de umbral")
print("Tiempo de ejecucion - Iteracion 5 (segundos):", round(tiempo_ejecucion_iteracion_5, 2))
print("Mejor score combinado promedio en CV:", round(score_combinado_i5, 4))
print("Mejor configuracion encontrada:")
print(random_search_iteracion_5.best_params_)

resumen_busqueda_iteracion_5 = pd.DataFrame({
    "elemento": ["combinaciones_evaluadas", "folds_cv", "total_ajustes_cv", "tiempo_ejecucion_segundos", "mejor_score_combinado_cv"],
    "valor": [n_iter_iteracion_5, n_folds_iteracion_5, n_iter_iteracion_5 * n_folds_iteracion_5, round(tiempo_ejecucion_iteracion_5, 2), round(score_combinado_i5, 4)]
})
display(resumen_busqueda_iteracion_5)

resumen_cv_iteracion_5 = pd.DataFrame({
    "metrica_cv": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "mejor_modelo_cv": [
        cv_results_i5["mean_test_accuracy"][best_index_iteracion_5],
        cv_results_i5["mean_test_precision"][best_index_iteracion_5],
        cv_results_i5["mean_test_recall"][best_index_iteracion_5],
        cv_results_i5["mean_test_f1"][best_index_iteracion_5],
        cv_results_i5["mean_test_roc_auc"][best_index_iteracion_5]
    ]
})
resumen_cv_iteracion_5["mejor_modelo_cv"] = resumen_cv_iteracion_5["mejor_modelo_cv"].round(4)
display(resumen_cv_iteracion_5)

# Ajuste de umbral con train out-of-fold. No usa test para seleccionar umbral.
best_params_iteracion_5 = random_search_iteracion_5.best_params_
pipeline_umbral_iteracion_5 = clone(formal_model_pipeline_template).set_params(**best_params_iteracion_5)

oof_proba_iteracion_5 = cross_val_predict(
    pipeline_umbral_iteracion_5,
    X_train_formal,
    y_train_formal,
    cv=cv_iteracion_5,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

threshold_candidates_iteracion_5 = np.arange(0.15, 0.81, 0.01)
roc_auc_oof_i5 = roc_auc_score(y_train_formal, oof_proba_iteracion_5)
threshold_results_iteracion_5 = []
for threshold in threshold_candidates_iteracion_5:
    y_oof_pred = (oof_proba_iteracion_5 >= threshold).astype(int)
    threshold_results_iteracion_5.append({
        "threshold": threshold,
        "accuracy_cv_oof": accuracy_score(y_train_formal, y_oof_pred),
        "precision_cv_oof": precision_score(y_train_formal, y_oof_pred, zero_division=0),
        "recall_cv_oof": recall_score(y_train_formal, y_oof_pred, zero_division=0),
        "f1_cv_oof": f1_score(y_train_formal, y_oof_pred, zero_division=0),
        "roc_auc_cv_oof": roc_auc_oof_i5
    })

threshold_results_iteracion_5 = pd.DataFrame(threshold_results_iteracion_5)
precision_minima_iteracion_5 = 0.35
candidatos_umbral_i5 = threshold_results_iteracion_5[threshold_results_iteracion_5["precision_cv_oof"] >= precision_minima_iteracion_5]

if candidatos_umbral_i5.empty:
    best_threshold_row_iteracion_5 = threshold_results_iteracion_5.sort_values(["f1_cv_oof", "precision_cv_oof"], ascending=False).iloc[0]
else:
    best_threshold_row_iteracion_5 = candidatos_umbral_i5.sort_values(["f1_cv_oof", "precision_cv_oof", "recall_cv_oof"], ascending=False).iloc[0]

best_threshold_iteracion_5 = best_threshold_row_iteracion_5["threshold"]
print("Precision minima exigida para seleccionar umbral:", precision_minima_iteracion_5)
print("Mejor umbral elegido solo con train/out-of-fold:", round(best_threshold_iteracion_5, 2))
display(threshold_results_iteracion_5.sort_values("f1_cv_oof", ascending=False).head(10).round(4))

best_pipeline_iteracion_5 = clone(formal_model_pipeline_template).set_params(**best_params_iteracion_5)
best_pipeline_iteracion_5.fit(X_train_formal, y_train_formal)

y_iteracion_5_proba = best_pipeline_iteracion_5.predict_proba(X_test_formal)[:, 1]
y_iteracion_5_pred_default = best_pipeline_iteracion_5.predict(X_test_formal)
y_iteracion_5_pred_threshold = (y_iteracion_5_proba >= best_threshold_iteracion_5).astype(int)

metricas_test_iteracion_5 = pd.DataFrame({
    "metrica_test": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "umbral_0_50": [
        accuracy_score(y_test_formal, y_iteracion_5_pred_default),
        precision_score(y_test_formal, y_iteracion_5_pred_default, zero_division=0),
        recall_score(y_test_formal, y_iteracion_5_pred_default, zero_division=0),
        f1_score(y_test_formal, y_iteracion_5_pred_default, zero_division=0),
        roc_auc_score(y_test_formal, y_iteracion_5_proba)
    ],
    "umbral_ajustado": [
        accuracy_score(y_test_formal, y_iteracion_5_pred_threshold),
        precision_score(y_test_formal, y_iteracion_5_pred_threshold, zero_division=0),
        recall_score(y_test_formal, y_iteracion_5_pred_threshold, zero_division=0),
        f1_score(y_test_formal, y_iteracion_5_pred_threshold, zero_division=0),
        roc_auc_score(y_test_formal, y_iteracion_5_proba)
    ]
})
metricas_test_iteracion_5[["umbral_0_50", "umbral_ajustado"]] = metricas_test_iteracion_5[["umbral_0_50", "umbral_ajustado"]].round(4)
display(metricas_test_iteracion_5)

print("Matriz de confusion - Iteracion 5 con umbral ajustado:")
print(confusion_matrix(y_test_formal, y_iteracion_5_pred_threshold))
print("Reporte de clasificacion - Iteracion 5 con umbral ajustado:")
print(classification_report(y_test_formal, y_iteracion_5_pred_threshold, zero_division=0))

resultados_random_search_iteracion_5 = pd.DataFrame(random_search_iteracion_5.cv_results_)
resultados_random_search_iteracion_5["score_combinado"] = (
    0.38 * resultados_random_search_iteracion_5["mean_test_f1"] +
    0.32 * resultados_random_search_iteracion_5["mean_test_roc_auc"] +
    0.24 * resultados_random_search_iteracion_5["mean_test_precision"] +
    0.06 * resultados_random_search_iteracion_5["mean_test_recall"]
)
resultados_random_search_iteracion_5 = resultados_random_search_iteracion_5.sort_values("score_combinado", ascending=False)

display(resultados_random_search_iteracion_5[
    ["score_combinado", "mean_test_f1", "std_test_f1", "mean_test_precision", "mean_test_recall", "mean_test_roc_auc", "mean_test_accuracy",
     "param_feature_selection__threshold", "param_dimensionality_reduction__n_components", "param_training__C", "param_training__penalty",
     "param_training__solver", "param_training__class_weight", "param_training__max_iter"]
].head(10))

metricas_previas_i5 = obtener_metricas_previas("metricas_test_iteracion_3")
if metricas_previas_i5 is not None:
    comparacion_i3_i5 = metricas_previas_i5.merge(
        metricas_test_iteracion_5[["metrica_test", "umbral_ajustado"]].rename(columns={"umbral_ajustado": "iteracion_5"}),
        on="metrica_test",
        how="inner"
    )
    comparacion_i3_i5["diferencia_i5_menos_anterior"] = (comparacion_i3_i5["iteracion_5"] - comparacion_i3_i5["iteracion_anterior"]).round(4)
    display(comparacion_i3_i5)
else:
    print("No se encontro metricas_test_iteracion_3 en memoria; ejecute Iteracion 3 para ver la comparacion directa.")


Espacio de busqueda utilizado - Iteracion 5


,hiperparametro,valores_explorados
0,feature_selection__threshold,"[0.0, 0.001, 0.003, 0.005, 0.0075, 0.01]"
1,dimensionality_reduction__n_components,"[0.875, 0.9, 0.925, 0.95, 0.975]"
2,training__C,<scipy.stats._distn_infrastructure.rv_continuo...
3,training__penalty,"['l1', 'l2']"
4,training__solver,"['liblinear', 'saga']"
5,training__class_weight,"[None, 'balanced']"
6,training__max_iter,"[8000, 10000]"


Cantidad de combinaciones evaluadas: 120
Cantidad de folds CV: 5
Total de ajustes realizados: 600
Fitting 5 folds for each of 120 candidates, totalling 600 fits
ITERACION 5 - RandomizedSearchCV + ajuste de umbral
Tiempo de ejecucion - Iteracion 5 (segundos): 146.95
Mejor score combinado promedio en CV: 0.5005
Mejor configuracion encontrada:
{'dimensionality_reduction__n_components': 0.975, 'feature_selection__threshold': 0.0075, 'training__C': np.float64(0.003974043077085724), 'training__class_weight': None, 'training__max_iter': 8000, 'training__penalty': 'l2', 'training__solver': 'liblinear'}


,elemento,valor
0,combinaciones_evaluadas,120.0000
1,folds_cv,5.0000
2,total_ajustes_cv,600.0000
3,tiempo_ejecucion_segundos,146.9500
4,mejor_score_combinado_cv,0.5005


,metrica_cv,mejor_modelo_cv
0,Accuracy,0.8823
1,Precision,0.5310
2,Recall,0.2250
3,F1 Score,0.2816
4,ROC-AUC,0.7891


...

,threshold,accuracy_cv_oof,precision_cv_oof,recall_cv_oof,f1_cv_oof,roc_auc_cv_oof
30,0.45,0.8824,0.4688,0.375,0.4167,0.7794
29,0.44,0.8655,0.4000,0.400,0.4000,0.7794
28,0.43,0.8571,0.3721,0.400,0.3855,0.7794
27,0.42,0.8403,0.3333,0.425,0.3736,0.7794
31,0.46,0.8768,0.4333,0.325,0.3714,0.7794
24,0.39,0.7423,0.2547,0.675,0.3699,0.7794
26,0.41,0.8235,0.3051,0.450,0.3636,0.7794
32,0.47,0.8796,0.4444,0.300,0.3582,0.7794
23,0.38,0.6611,0.2245,0.825,0.3529,0.7794
33,0.48,0.8852,0.4783,0.275,0.3492,0.7794


,metrica_test,umbral_0_50,umbral_ajustado
0,Accuracy,0.8655,0.8571
1,Precision,0.2000,0.2500
2,Recall,0.0769,0.1538
3,F1 Score,0.1111,0.1905
4,ROC-AUC,0.8476,0.8476


Matriz de confusion - Iteracion 5 con umbral ajustado:
[[100   6]
 [ 11   2]]
Reporte de clasificacion - Iteracion 5 con umbral ajustado:
              precision    recall  f1-score   support

           0       0.90      0.94      0.92       106
           1       0.25      0.15      0.19        13

    accuracy                           0.86       119
   macro avg       0.58      0.55      0.56       119
weighted avg       0.83      0.86      0.84       119



,score_combinado,mean_test_f1,std_test_f1,mean_test_precision,mean_test_recall,mean_test_roc_auc,mean_test_accuracy,param_feature_selection__threshold,param_dimensionality_reduction__n_components,param_training__C,param_training__penalty,param_training__solver,param_training__class_weight,param_training__max_iter
15,0.500460,0.281587,0.131743,0.530952,0.225,0.789149,0.882277,0.0075,0.975,0.003974,l2,liblinear,None,8000
34,0.482925,0.254141,0.124321,0.566667,0.175,0.749535,0.887872,0.0030,0.950,0.060207,l2,saga,None,8000
13,0.482800,0.254141,0.124321,0.566667,0.175,0.749144,0.887872,0.0000,0.950,13.986791,l1,saga,None,10000
4,0.482800,0.254141,0.124321,0.566667,0.175,0.749144,0.887872,0.0000,0.950,0.029238,l2,saga,None,8000
41,0.482800,0.254141,0.124321,0.566667,0.175,0.749144,0.887872,0.0050,0.950,0.022098,l2,saga,None,10000
52,0.482800,0.254141,0.124321,0.566667,0.175,0.749144,0.887872,0.0100,0.950,13.084114,l1,saga,None,10000
98,0.482800,0.254141,0.124321,0.566667,0.175,0.749144,0.887872,0.0000,0.950,12.303376,l2,saga,None,8000
77,0.473440,0.338337,0.125981,0.226895,0.750,0.766927,0.626526,0.0050,0.975,0.581346,l2,liblinear,balanced,8000
19,0.469785,0.330693,0.098652,0.215941,0.775,0.768111,0.618192,0.0030,0.975,0.463683,l1,liblinear,balanced,8000
29,0.465336,0.254141,0.124321,0.566667,0.175,0.694568,0.887872,0.0010,0.925,0.044822,l2,saga,None,8000


,metrica_test,iteracion_anterior,iteracion_5,diferencia_i5_menos_anterior
0,Accuracy,0.8571,0.8571,0.0000
1,Precision,0.3333,0.2500,-0.0833
2,Recall,0.3077,0.1538,-0.1539
3,F1 Score,0.3200,0.1905,-0.1295
4,ROC-AUC,0.8215,0.8476,0.0261
